In [ ]:
"""
MSI Delocalization Scoring Pipeline
────────────────────────────────────
Quantifies analyte delocalization from tissue into the background region
using spatial metrics (mean distance, nonzero area) and produces ranked
heatmap visualisations for every m/z feature.

Workflow:
  1. Load overlaid h5ad (with tissue mask from the annotation tool)
  2. TIC-normalise intensities
  3. Remove m/z features that are not enriched in tissue (fold-change filter)
  4. Mask weak background signals (below a fraction of tissue max)
  5. Compute delocalization metrics per m/z (mean BG-to-tissue distance, area)
  6. Score and rank m/z features
  7. Store metrics in adata.var + export ranked heatmap images

Folder structure (relative to the working directory):
  <working_dir>/
  ├── overlaid_h5ad/                ← input  (h5ad with tissue column)
  └── results/                      ← output (h5ad with scores + image subfolder)
"""

import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial import cKDTree
from skimage.measure import find_contours
from tqdm import tqdm

matplotlib.use("Qt5Agg")  # Required for VSCode interactivity

# ═══════════════════════════════════════════════════════════════════════════════
# RESOLVE PATHS RELATIVE TO The WORKING DIRECTORY
# ═══════════════════════════════════════════════════════════════════════════════
WORKING_DIR = "/home/ajarrah/PhD_Thesis/Delocalization_Score_2"
OVERLAID_DIR = os.path.join(WORKING_DIR, "overlaid_h5ad")
RESULTS_DIR = os.path.join(WORKING_DIR, "results")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# File names
INPUT_FILENAME = "OFM_AD_3_overlaid.h5ad"
OUTPUT_FILENAME = "OFM_AD_3_scored.h5ad"

# Full paths (assembled automatically)
INPUT_FILE = os.path.join(OVERLAID_DIR, INPUT_FILENAME)
OUTPUT_FILE = os.path.join(RESULTS_DIR, OUTPUT_FILENAME)

# Tissue enrichment filter
TISSUE_FOLD_CHANGE = 1  # keep m/z where mean_tissue >= FC × mean_background

# Background intensity masking
# Set BG pixel to 0 if intensity < this fraction of tissue max for that m/z
MIN_BG_INTENSITY_FRAC = 0.1

# Delocalization score weighting
AREA_WEIGHT = 0.95  # weight for normalised area (1 - weight → mean distance)

# Custom colourmap (dark → navy → blue → purple → red → orange → yellow → white)
CUSTOM_CMAP = LinearSegmentedColormap.from_list("msi_heatmap", [
    (0.00,       "#454545"),
    (0.00000001, "#000000"),
    (0.10,       "#000080"),
    (0.15,       "#0000FF"),
    (0.30,       "#8000FF"),
    (0.45,       "#FF0000"),
    (0.60,       "#FF8000"),
    (0.75,       "#FFFF00"),
    (1.00,       "#FFFFFF"),
])


# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def _to_dense_col(X, col_idx):
    """Extract a single column from X as a dense 1-D array."""
    if sp.issparse(X):
        return X[:, col_idx].toarray().ravel()
    return np.asarray(X[:, col_idx]).ravel()


def tic_normalise(adata):
    """
    Total Ion Current normalisation (in-place).

    Stores original intensities in adata.layers["raw"] and per-pixel
    TIC values in adata.obs["tic"].
    """
    tic = np.asarray(adata.X.sum(axis=1)).ravel()
    tic[tic == 0] = 1.0

    adata.layers["raw"] = adata.X.copy()

    if sp.issparse(adata.X):
        adata.X = adata.X.multiply(1.0 / tic[:, np.newaxis])
    else:
        adata.X = adata.X / tic[:, np.newaxis]

    adata.obs["tic"] = tic
    print(f"  TIC normalisation applied ({adata.n_obs} pixels)")


def filter_mz_by_tissue(adata, fold_change=2.0):
    """
    Remove m/z features whose mean tissue intensity is below
    `fold_change × mean_background` intensity.
    """
    tissue = adata.obs["tissue"] == 1
    bg = adata.obs["tissue"] == 0

    mean_tissue = np.asarray(adata.X[tissue].mean(axis=0)).ravel()
    mean_bg = np.asarray(adata.X[bg].mean(axis=0)).ravel()

    keep = mean_tissue >= fold_change * mean_bg
    n_drop = (~keep).sum()
    print(f"  Dropping {n_drop} m/z features (tissue < {fold_change}× background)")
    return adata[:, keep].copy()


def mask_low_bg_intensities(adata, frac=0.1):
    """
    Zero out background pixels whose intensity is below
    `frac × max_tissue_intensity` for each m/z.
    """
    X = adata.X
    tissue_mask = adata.obs["tissue"].values == 1
    bg_indices = np.where(~tissue_mask)[0]

    if sp.issparse(X):
        X = X.tocsr()
        max_tissue = X[tissue_mask].max(axis=0).toarray().ravel()
    else:
        max_tissue = X[tissue_mask].max(axis=0)

    thresholds = frac * max_tissue

    for i in bg_indices:
        if sp.issparse(X):
            row = X[i].tocoo()
            row.data[row.data <= thresholds[row.col]] = 0
            X[i] = row.tocsr()
        else:
            below = X[i] <= thresholds
            X[i, below] = 0

    adata.X = X
    print(f"  Masked BG intensities < {frac}× tissue max ({len(bg_indices)} BG pixels)")


# ═══════════════════════════════════════════════════════════════════════════════
# DELOCALIZATION METRICS
# ═══════════════════════════════════════════════════════════════════════════════

def compute_delocalization_metrics(adata):
    """
    For each m/z, compute:
      - bg_dist_mean      : mean distance from nonzero BG pixels to nearest tissue pixel
      - bg_nonzero_area   : number of nonzero BG pixels

    Returns
    -------
    pd.DataFrame with one row per m/z
    """
    mz_values = adata.var["mzs"].values
    n_mz = len(mz_values)

    # Pre-compute tissue / background coordinate arrays
    tissue_mask = adata.obs["tissue"].values == 1
    bg_mask = ~tissue_mask

    tissue_coords = np.column_stack([
        adata.obs.loc[tissue_mask, "x"].values,
        adata.obs.loc[tissue_mask, "y"].values,
    ])
    bg_x = adata.obs.loc[bg_mask, "x"].values
    bg_y = adata.obs.loc[bg_mask, "y"].values

    # Build KD-tree for tissue pixels (shared across all m/z)
    tree = cKDTree(tissue_coords) if tissue_coords.shape[0] > 0 else None

    rows = []
    for j in tqdm(range(n_mz), desc="Computing metrics"):
        intensities = _to_dense_col(adata.X, j)
        bg_int = intensities[bg_mask]

        row = {"mz": mz_values[j]}

        # Nonzero background pixels
        valid = bg_int > 0
        n_valid = valid.sum()
        row["bg_nonzero_area"] = n_valid

        if n_valid > 0 and tree is not None:
            valid_coords = np.column_stack([bg_x[valid], bg_y[valid]])
            distances, _ = tree.query(valid_coords)
            row["bg_dist_mean"] = distances.mean()
        else:
            row["bg_dist_mean"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════════
# DELOCALIZATION SCORING
# ═══════════════════════════════════════════════════════════════════════════════

def score_delocalization(df, area_weight=0.95):
    """
    Add normalised columns and a weighted delocalization score.

    score = area_weight × norm_area + (1 − area_weight) × norm_mean_dist
    """
    def _minmax(s):
        lo, hi = s.min(), s.max()
        return (s - lo) / (hi - lo) if hi > lo else pd.Series(0.0, index=s.index)

    df["norm_bg_nonzero_area"] = _minmax(df["bg_nonzero_area"])
    df["norm_bg_dist_mean"] = _minmax(df["bg_dist_mean"])
    df["delocalization_score"] = (
        area_weight * df["norm_bg_nonzero_area"]
        + (1 - area_weight) * df["norm_bg_dist_mean"]
    )
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════════════════════════

def _tissue_contours(adata):
    """Compute tissue boundary contours from the binary tissue mask."""
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    tissue = adata.obs["tissue"].values == 1

    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_map = {v: i for i, v in enumerate(grid_x)}
    y_map = {v: i for i, v in enumerate(grid_y)}

    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)
    for xi, yi, t in zip(x, y, tissue):
        if t:
            mask_2d[y_map[yi], x_map[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)
    return [(grid_x[c[:, 1].astype(int)], grid_y[c[:, 0].astype(int)]) for c in contours]


def visualise_mz_heatmap(adata, mz_value, contours, cmap=None, save_path=None):
    """
    Plot heatmap for a single m/z with tissue contour overlay.

    Parameters
    ----------
    adata : AnnData
    mz_value : float
    contours : list of (cx, cy) arrays  – from _tissue_contours()
    cmap : colormap (default: CUSTOM_CMAP)
    save_path : str or None – if provided, save figure and close
    """
    if cmap is None:
        cmap = CUSTOM_CMAP

    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = _to_dense_col(adata.X, mz_idx)

    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(x, y, c=intensities, s=12, marker="s", cmap=cmap)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(f"Intensity @ m/z {mz_value:.2f}", fontsize=18)
    cbar.ax.tick_params(labelsize=16)

    for cx, cy in contours:
        ax.plot(cx, cy, color="red", linewidth=2)

    ax.set_title(f"m/z {mz_value:.2f}", fontsize=20)
    ax.set_xlabel("x", fontsize=18)
    ax.set_ylabel("y", fontsize=18)
    ax.tick_params(labelsize=16)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=300)
        plt.close(fig)
    else:
        plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    # --- Load ---
    print(f"Loading: {INPUT_FILE}")
    adata = sc.read_h5ad(INPUT_FILE)

    # --- TIC normalise ---
    tic_normalise(adata)

    # --- Filter m/z by tissue enrichment ---
    adata = filter_mz_by_tissue(adata, fold_change=TISSUE_FOLD_CHANGE)

    # --- Mask weak background signals ---
    mask_low_bg_intensities(adata, frac=MIN_BG_INTENSITY_FRAC)

    # --- Compute delocalization metrics ---
    df_metrics = compute_delocalization_metrics(adata)

    # --- Score and rank ---
    df_metrics = score_delocalization(df_metrics, area_weight=AREA_WEIGHT)
    df_metrics = df_metrics.sort_values("delocalization_score", ascending=False)

    # --- Store metrics in adata.var ---
    # Align metrics DataFrame to adata.var by m/z
    df_metrics = df_metrics.set_index("mz")
    for col in df_metrics.columns:
        adata.var[col] = df_metrics[col].values

    print(f"  Stored metrics in adata.var: {list(df_metrics.columns)}")

    # --- Save scored AnnData ---
    os.makedirs(RESULTS_DIR, exist_ok=True)
    adata.write(OUTPUT_FILE)
    print(f"  Saved: {OUTPUT_FILE}  ({adata.shape})")

    # --- Export ranked heatmaps ---
    df_sorted = df_metrics.sort_values("delocalization_score", ascending=False)
    image_dir = os.path.join(
        RESULTS_DIR,
        f"ranked_heatmaps_{MIN_BG_INTENSITY_FRAC}_score_{AREA_WEIGHT}",
    )
    os.makedirs(image_dir, exist_ok=True)

    contours = _tissue_contours(adata)
    for rank, (mz_val, _) in enumerate(df_sorted.iterrows(), start=1):
        path = os.path.join(image_dir, f"{rank}_mz_{mz_val:.4f}.png")
        visualise_mz_heatmap(adata, mz_val, contours, save_path=path)

    print(f"  Saved {len(df_sorted)} ranked heatmaps → {image_dir}")


if __name__ == "__main__":
    main()


In [1]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc
import os
import scipy.sparse as sp
import anndata as ad
import pandas as pd
from scipy.spatial import cKDTree
from tqdm import tqdm
from skimage.measure import find_contours

In [2]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/h5ad_data_overlaid/OFM_AD_3_overlaid.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/h5ad_data_overlaid/OFM_AD_3_overlaid.h5ad


In [3]:
batch = adata.obs['batch'][0] 
sample = adata.obs['sample'][0]
adata

/tmp/ipykernel_3314702/3043734248.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  batch = adata.obs['batch'][0]
/tmp/ipykernel_3314702/3043734248.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sample = adata.obs['sample'][0]


AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [4]:
sample_group = adata.obs['age_group'][0]+"_" + adata.obs['disease_status'][0]
sample_group

/tmp/ipykernel_3314702/2201400382.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sample_group = adata.obs['age_group'][0]+"_" + adata.obs['disease_status'][0]


'Aged_AD'

In [5]:

def tic_normalize(adata, inplace = True, layer_name = None):
    """
    Perform Total Ion Current (TIC) normalization on an AnnData object.
    
    Parameters:
    - adata (AnnData): The input AnnData object with intensity matrix in `.X`.
    - inplace (bool): If True, overwrite `adata.X`. If False, store result in `adata.layers[layer_name]`.
    - layer_name (str): Name of the layer to store normalized matrix. Required if `inplace=False`.

    Returns:
    - None. The AnnData object is modified in-place.
    """

    if not inplace and not layer_name:
        raise ValueError("You must specify 'layer_name' when inplace=False.")

    # Compute TIC per observation
    tic = adata.X.sum(axis=1)
    tic = np.asarray(tic).flatten()
    tic[tic == 0] = 1.0  # Avoid division by zero

    # Normalize
    if sp.issparse(adata.X):
        normalized = adata.X.multiply(1 / tic[:, np.newaxis])
    else:
        normalized = adata.X / tic[:, np.newaxis]

    # Store result
    if inplace:
        adata.layers["raw"] = adata.X.copy()  # Store raw values before normalization
        adata.X = normalized
    else:
        adata.layers[layer_name] = normalized

    # Save TIC to .obs
    adata.obs["tic"] = tic

    print(f"✅ TIC normalization complete. {'Updated adata.X' if inplace else f'Stored in layer: {layer_name}'}")
    

In [6]:
tic_normalize(adata, inplace=True)

✅ TIC normalization complete. Updated adata.X


In [7]:
def filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=2.0, layer=None, normalize_tic=True):
    # 1. Extract raw data
    data = adata.layers[layer] if layer is not None else adata.X

    # 2. TIC normalization per pixel
    if normalize_tic:
        # Sum across features for each observation (axis=1)
        tic = np.asarray(data.sum(axis=1)).ravel()
        # Avoid division by zero
        tic[tic == 0] = 1.0
        # Normalize in place (if sparse, convert to array)
        data = data / tic[:, None]

    # 3. Define tissue masks
    mask_tissue = adata.obs[tissue_key] == tissue_val
    mask_non    = adata.obs[tissue_key] == non_tissue_val

    # 4. Compute mean normalized intensity per feature
    mean_tissue    = np.asarray(data[mask_tissue].mean(axis=0)).ravel()
    mean_nontissue = np.asarray(data[mask_non].mean(axis=0)).ravel()

    # 5. Identify features to remove
    to_remove = mean_tissue < foldchange * mean_nontissue

    # 6. Print removed m/z names
    mz_to_remove = adata.var_names[to_remove]
    print(f"Dropping {len(mz_to_remove)} m/z features:", mz_to_remove.values)

    # 7. Subset and return
    return adata[:, ~to_remove].copy()

In [8]:
foldchange = 1

# Filter m/z features based on tissue mask
adata = filter_mz_by_tissue(adata, tissue_key="tissue", tissue_val=1, non_tissue_val=0, foldchange=foldchange, layer=None, normalize_tic=False)

Dropping 590 m/z features: ['136.0150' '137.0251' '138.0287' '139.0359' '154.0255' '155.0350'
 '156.0424' '159.0070' '160.0107' '163.0405' '165.0200' '172.0540'
 '173.0578' '176.0111' '177.0146' '178.0269' '181.0163' '187.0399'
 '189.0757' '190.0676' '191.0379' '192.9801' '195.0905' '196.0981'
 '196.9856' '197.1082' '197.9919' '199.0008' '199.0439' '199.9998'
 '200.0492' '201.0571' '202.0613' '202.1793' '203.0369' '204.0398'
 '210.9897' '211.0595' '213.0560' '214.0577' '215.0298' '216.0426'
 '217.0514' '218.0560' '219.0630' '219.9751' '220.9800' '227.0378'
 '228.0455' '229.0025' '229.0488' '230.0543' '231.0686' '232.0653'
 '233.0507' '239.0343' '240.0412' '241.0502' '243.0269' '244.0353'
 '245.0458' '246.0516' '247.0594' '248.0624' '249.0743' '251.0421'
 '254.0257' '255.0287' '256.0338' '257.0478' '258.0498' '259.0608'
 '260.0386' '261.0393' '262.0491' '263.0537' '264.0603' '265.0616'
 '267.0272' '268.0413' '269.0502' '270.0538' '271.0377' '272.0306'
 '273.0397' '273.2201' '274.0435' '

In [9]:
adata.var_names[0]

'136.0608'

In [10]:
def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, export_border_points=False):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    all_border_data = []
    rows = []

    tissue_centroid_x = adata.obs.loc[adata.obs["tissue"] == True, "x"].mean()
    tissue_centroid_y = adata.obs.loc[adata.obs["tissue"] == True, "y"].mean()

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        # Global center of mass and distance to tissue centroid
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)

        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - tissue_centroid_x) ** 2 + (y_com - tissue_centroid_y) ** 2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        # Background distance to tissue border for nonzero-intensity pixels
        bg_subset = adata[adata.obs["tissue"] == False]
        tissue_subset = adata[adata.obs["tissue"] == True]

        bg_x = bg_subset.obs["x"].values
        bg_y = bg_subset.obs["y"].values
        bg_intensities = bg_subset.X[:, mz_index].toarray().flatten() if hasattr(bg_subset.X, "toarray") else bg_subset.X[:, mz_index].flatten()

        tissue_x = tissue_subset.obs["x"].values
        tissue_y = tissue_subset.obs["y"].values

        if len(bg_intensities) > 0 and len(tissue_x) > 0:
            valid_mask = bg_intensities > 0
            valid_bg_coords = np.vstack([bg_x[valid_mask], bg_y[valid_mask]]).T
            tissue_coords = np.vstack([tissue_x, tissue_y]).T

            if valid_bg_coords.shape[0] > 0:
                tree = cKDTree(tissue_coords)
                distances, indices = tree.query(valid_bg_coords)

                nearest_tissue_points = tissue_coords[indices]

                row["bg_dist_max"] = np.max(distances)
                row["bg_dist_mean"] = np.mean(distances)
                row["bg_nonzero_area"] = valid_bg_coords.shape[0]

                if export_border_points:
                    for i in range(len(valid_bg_coords)):
                        all_border_data.append({
                            "mz": mz_values[mz_index],
                            "bg_x": valid_bg_coords[i, 0],
                            "bg_y": valid_bg_coords[i, 1],
                            "tissue_x": nearest_tissue_points[i, 0],
                            "tissue_y": nearest_tissue_points[i, 1],
                            "distance": distances[i]
                        })

            else:
                row["bg_dist_max"] = np.nan
                row["bg_dist_mean"] = np.nan
                row["bg_nonzero_area"] = 0
        else:
            row["bg_dist_max"] = np.nan
            row["bg_dist_mean"] = np.nan
            row["bg_nonzero_area"] = 0

        # Weighted centroids
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()
        tissue_centroid_x_weighted, tissue_centroid_y_weighted = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)
        row["weighted_com_x_tissue"] = tissue_centroid_x_weighted
        row["weighted_com_y_tissue"] = tissue_centroid_y_weighted
        row["weighted_com_x_all"] = x_com
        row["weighted_com_y_all"] = y_com

        rows.append(row)

    df_metrics = pd.DataFrame(rows)

    if export_border_points:
        df_border = pd.DataFrame(all_border_data)
        return df_metrics, df_border
    else:
        return df_metrics

In [11]:
pre_df_metrics, pre_df_border = analyze_metrics(adata, export_border_points=True)

Processing m/z features: 100%|██████████| 410/410 [00:15<00:00, 26.03it/s]


In [12]:
def mask_low_background_intensities(adata, fold_change=0.2):
    """
    Set background pixel intensities (tissue==0) to 0 if they are below fold_change * max tissue intensity per m/z.
    
    Parameters:
        adata : AnnData
            An AnnData object where .X is (n_pixels x n_mz) and .obs["tissue"] exists.
        fold_change : float
            The threshold multiplier to compare background intensity to max tissue intensity.
    """
    X = adata.X
    tissue_mask = adata.obs["tissue"] == 1
    background_mask = ~tissue_mask

    if sp.issparse(X):
        X = X.tocsr()

    # Compute max intensity of each m/z in tissue pixels
    max_tissue_intensity = X[tissue_mask.values].max(axis=0).A1 if sp.issparse(X) else X[tissue_mask.values].max(axis=0)

    # Define threshold per m/z
    thresholds = fold_change * max_tissue_intensity

    # Modify background pixels
    bg_indices = np.where(background_mask)[0]

    for i in bg_indices:
        row = X[i]
        if sp.issparse(X):
            row = row.tocoo()
            mask = row.data <= thresholds[row.col]
            row.data[mask] = 0
            X[i] = sp.csr_matrix((row.data, (np.zeros_like(row.col), row.col)), shape=(1, X.shape[1]))
        else:
            mask = row <= thresholds
            row[mask] = 0
            X[i] = row

    adata.X = X
    return adata

In [13]:
min_background_intensity = 0.1
adata = mask_low_background_intensities(adata, fold_change=min_background_intensity)

In [14]:

def compute_center_of_mass(x_coords, y_coords, intensities):
    total_intensity = np.sum(intensities)
    if total_intensity == 0:
        return np.nan, np.nan
    x_center = np.sum(x_coords * intensities) / total_intensity
    y_center = np.sum(y_coords * intensities) / total_intensity
    return x_center, y_center


def analyze_metrics(adata, export_border_points=False):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    all_border_data = []
    rows = []

    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):

        # Initialize row for current m/z
        row = {"mz": mz_values[mz_index]}

        # Extract x, y, and intensity for current mz
        all_x = adata.obs["x"].values
        all_y = adata.obs["y"].values
        all_intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index].flatten()

        # Compute global center of mass
        x_com, y_com = compute_center_of_mass(all_x, all_y, all_intensities)

        # Only tissue pixels
        tissue_mask = adata.obs["tissue"].values == True
        tissue_x = all_x[tissue_mask]
        tissue_y = all_y[tissue_mask]
        tissue_intensities = all_intensities[tissue_mask]

        # Compute weighted tissue centroid (center of mass within tissue)
        x_com_tissue, y_com_tissue = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)

        # Compute distance from global COM to tissue COM
        if not np.isnan(x_com) and not np.isnan(y_com):
            dist_to_tissue_centroid = np.sqrt((x_com - x_com_tissue) ** 2 + (y_com - y_com_tissue) ** 2)
        else:
            dist_to_tissue_centroid = np.nan

        row["com_distance_to_tissue"] = dist_to_tissue_centroid

        # Background distance to tissue border for nonzero-intensity pixels
        bg_subset = adata[adata.obs["tissue"] == False]
        tissue_subset = adata[adata.obs["tissue"] == True]

        bg_x = bg_subset.obs["x"].values
        bg_y = bg_subset.obs["y"].values
        bg_intensities = bg_subset.X[:, mz_index].toarray().flatten() if hasattr(bg_subset.X, "toarray") else bg_subset.X[:, mz_index].flatten()

        tissue_x = tissue_subset.obs["x"].values
        tissue_y = tissue_subset.obs["y"].values
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()

        # Add mean intensities
        row["mean_intensity_tissue"] = np.mean(tissue_intensities) if tissue_intensities.size > 0 else np.nan
        row["mean_intensity_bg"] = np.mean(bg_intensities) if bg_intensities.size > 0 else np.nan

        if len(bg_intensities) > 0 and len(tissue_x) > 0:
            valid_mask = bg_intensities > 0
            valid_bg_coords = np.vstack([bg_x[valid_mask], bg_y[valid_mask]]).T
            tissue_coords = np.vstack([tissue_x, tissue_y]).T

            if valid_bg_coords.shape[0] > 0:
                tree = cKDTree(tissue_coords)
                distances, indices = tree.query(valid_bg_coords)

                nearest_tissue_points = tissue_coords[indices]

                row["bg_dist_max"] = np.max(distances)
                row["bg_dist_mean"] = np.mean(distances)
                row["bg_nonzero_area"] = valid_bg_coords.shape[0]

                if export_border_points:
                    for i in range(len(valid_bg_coords)):
                        all_border_data.append({
                            "mz": mz_values[mz_index],
                            "bg_x": valid_bg_coords[i, 0],
                            "bg_y": valid_bg_coords[i, 1],
                            "tissue_x": nearest_tissue_points[i, 0],
                            "tissue_y": nearest_tissue_points[i, 1],
                            "distance": distances[i]
                        })

            else:
                row["bg_dist_max"] = np.nan
                row["bg_dist_mean"] = np.nan
                row["bg_nonzero_area"] = 0
        else:
            row["bg_dist_max"] = np.nan
            row["bg_dist_mean"] = np.nan
            row["bg_nonzero_area"] = 0

        # Weighted centroids
        tissue_intensities = tissue_subset.X[:, mz_index].toarray().flatten() if hasattr(tissue_subset.X, "toarray") else tissue_subset.X[:, mz_index].flatten()
        tissue_centroid_x_weighted, tissue_centroid_y_weighted = compute_center_of_mass(tissue_x, tissue_y, tissue_intensities)
        row["weighted_com_x_tissue"] = tissue_centroid_x_weighted
        row["weighted_com_y_tissue"] = tissue_centroid_y_weighted
        row["weighted_com_x_all"] = x_com
        row["weighted_com_y_all"] = y_com

        rows.append(row)

    df_metrics = pd.DataFrame(rows)

    if export_border_points:
        df_border = pd.DataFrame(all_border_data)
        return df_metrics, df_border
    else:
        return df_metrics

In [15]:
import matplotlib.pyplot as plt

def visualize_distance_map(df_border, mz_value, n_arrows=100):
    subset = df_border[df_border["mz"] == mz_value]
    if subset.empty:
        print(f"No border data found for m/z {mz_value}")
        return

    fig, ax = plt.subplots(figsize=(12, 12))  # square figure
    ax.scatter(subset["bg_x"], subset["bg_y"], c="blue", s=8, label="Background Pixels")
    ax.scatter(subset["tissue_x"], subset["tissue_y"], c="red", s=8, label="Nearest Tissue Border")

    # Draw arrows from bg pixel to nearest tissue
    sample = subset.sample(n=min(n_arrows, len(subset)))
    for _, row in sample.iterrows():
        ax.arrow(row["bg_x"], row["bg_y"],
                 row["tissue_x"] - row["bg_x"],
                 row["tissue_y"] - row["bg_y"],
                 color="gray", alpha=0.8, head_width=0.5)
    
    mz_name = f"{mz_value:.2f}"
    ax.set_title(f"Nearest Tissue Border Points for m/z {mz_name}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.set_aspect("equal")       # maintain equal scale on both axes
    ax.invert_yaxis()            # flip vertically
    plt.tight_layout()
    plt.show()

In [16]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrow
from matplotlib.lines import Line2D

def visualize_distance_map(df_border, mz_value, n_arrows=100):
    subset = df_border[df_border["mz"] == mz_value]
    if subset.empty:
        print(f"No border data found for m/z {mz_value}")
        return

    fig, ax = plt.subplots(figsize=(8, 6))  # square figure
    bg_scatter = ax.scatter(subset["bg_x"], subset["bg_y"], c="blue", s=10, label="Background Pixels")
    tissue_scatter = ax.scatter(subset["tissue_x"], subset["tissue_y"], c="red", s=10, label="Nearest Tissue Border")

    # Draw arrows from bg pixel to nearest tissue
    sample = subset.sample(n=min(n_arrows, len(subset)))
    for _, row in sample.iterrows():
        ax.arrow(row["bg_x"], row["bg_y"],
                 row["tissue_x"] - row["bg_x"],
                 row["tissue_y"] - row["bg_y"],
                 color="gray", alpha=0.8, head_width=0.5)

    # Create a dummy arrow for legend
    arrow_legend = FancyArrow(0, 0, 1, 0, color="gray", alpha=1, width=0.1)

    # Custom legend handles
    legend_handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Background Pixels'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Nearest Tissue Border'),
        Line2D([0], [0], color='gray', lw=2, label='Direction (BG → Tissue)')
    ]

    mz_name = f"{mz_value:.2f}"
    ax.set_title(f"Nearest Tissue Border Points for m/z {mz_name}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(handles=legend_handles)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


In [17]:
df_metrics, df_border = analyze_metrics(adata, export_border_points=True)

Processing m/z features: 100%|██████████| 410/410 [00:14<00:00, 29.01it/s]


In [18]:
# Visualize arrows for m/z=885.55 (example)
visualize_distance_map(df_border, mz_value=651.5335, n_arrows=100)


MESA-LOADER: failed to open zink: /usr/lib/dri/zink_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)
MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)


In [19]:
def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot", save_path=None):
    from skimage.measure import find_contours
    import matplotlib.pyplot as plt
    import numpy as np

    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    if df_mz_border.empty:
        print(f"⚠️ Skipping m/z {mz_value:.4f} — no matching metric data found.")
    else:
        max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
        bg_point = (max_row["bg_x"], max_row["bg_y"])
        nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(3.7, 2.8))

    # Plot heatmap
    scatter = plt.scatter(x, y, c=intensities, s=12, marker='s', cmap=cmap)
    cbar = plt.colorbar(scatter)
    cbar.set_label(f"Intensity @ m/z {mz_value:.2f}", fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    # Plot COMs and max distance markers
    plt.scatter(*com_all, color="cyan", s=15, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=15, marker="*", label="COM (tissue)")
    if not df_mz_border.empty:
        plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]],
                 color="orange", linestyle="--", linewidth=0.5, label="Max dist to tissue")
        plt.scatter(*bg_point, color="red", s=15, edgecolor="black", label="Furthest BG pt")
        plt.scatter(*nearest_tissue_point, color="green", s=15, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue contour
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=1, label="Tissue border")

    # Make all labels bigger
   # plt.legend(fontsize=14)
    plt.title(f"m/z {mz_value:.2f}", fontsize=11)
    plt.xlabel("x", fontsize=10)
    plt.ylabel("y", fontsize=10)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()

    # Save or show
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

In [20]:
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

#visualize_mz_feature(adata, df_metrics, df_border, mz_value=float(adata.var_names[10]), cmap=custom_cmap, save_path=None)

In [21]:

def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot"):
    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
    bg_point = (max_row["bg_x"], max_row["bg_y"])
    nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot heatmap with square pixels
    plt.scatter(x, y, c=intensities, s=11, marker='s', cmap=cmap)
    plt.colorbar(label=f"Intensity @ m/z {mz_value:.2f}")

    # Plot center of mass
    plt.scatter(*com_all, color="cyan", s=150, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=150, marker="*", label="COM (tissue)")

    # Max distance point and nearest tissue point
    plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]], 
             color="orange", linestyle="--", label="Max dist to tissue")
    plt.scatter(*bg_point, color="red", s=80, edgecolor="black", label="Furthest BG pt")
    plt.scatter(*nearest_tissue_point, color="green", s=80, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue margin line
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=2, label="Tissue border")

    plt.legend()
    plt.title(f"m/z {mz_value:.2f} Heatmap with COMs and Furthest BG Point")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [22]:
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])
#custom_cmap = "binary"
visualize_mz_feature(adata, df_metrics, df_border, mz_value=float(adata.var_names[0]), cmap=custom_cmap)

In [23]:
def visualize_mz_feature(adata, df_metrics, df_border, mz_value, cmap="hot", save_path=None):
    from skimage.measure import find_contours
    import matplotlib.pyplot as plt
    import numpy as np

    # Find index of target m/z
    mz_idx = np.argmin(np.abs(adata.var["mzs"].values - mz_value))

    # Get coordinates and intensities
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    intensities = adata.X[:, mz_idx].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_idx].flatten()

    tissue_mask = adata.obs["tissue"].values

    # Extract COMs and border points
    mz_metrics = df_metrics[df_metrics["mz"] == mz_value].iloc[0]
    com_all = (mz_metrics["weighted_com_x_all"], mz_metrics["weighted_com_y_all"])
    com_tissue = (mz_metrics["weighted_com_x_tissue"], mz_metrics["weighted_com_y_tissue"])

    df_mz_border = df_border[df_border["mz"] == mz_value]
    if df_mz_border.empty:
        print(f"⚠️ Skipping m/z {mz_value:.4f} — no matching metric data found.")
    else:
        max_row = df_mz_border.loc[df_mz_border["distance"].idxmax()]
        bg_point = (max_row["bg_x"], max_row["bg_y"])
        nearest_tissue_point = (max_row["tissue_x"], max_row["tissue_y"])

    # Create 2D grid (image) for tissue mask
    grid_x = np.unique(x)
    grid_y = np.unique(y)
    x_to_idx = {v: i for i, v in enumerate(grid_x)}
    y_to_idx = {v: i for i, v in enumerate(grid_y)}
    mask_2d = np.zeros((len(grid_y), len(grid_x)), dtype=bool)

    for xi, yi, is_tissue in zip(x, y, tissue_mask):
        if is_tissue:
            mask_2d[y_to_idx[yi], x_to_idx[xi]] = True

    contours = find_contours(mask_2d.astype(float), level=0.5)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot heatmap
    scatter =plt.scatter(x, y, c=intensities, s=12, marker='s', cmap=cmap)
    cbar = plt.colorbar(scatter)
    cbar.set_label(f"Intensity @ m/z {mz_value:.2f}", fontsize=18)
    cbar.ax.tick_params(labelsize=16)

    # Plot COMs and max distance markers
    plt.scatter(*com_all, color="cyan", s=150, marker="*", label="COM (all)")
    plt.scatter(*com_tissue, color="lime", s=150, marker="*", label="COM (tissue)")
    if not df_mz_border.empty:
        plt.plot([bg_point[0], nearest_tissue_point[0]], [bg_point[1], nearest_tissue_point[1]], 
                color="orange", linestyle="--", linewidth=2, label="Max dist to tissue")
        plt.scatter(*bg_point, color="red", s=120, edgecolor="black", label="Furthest BG pt")
        plt.scatter(*nearest_tissue_point, color="green", s=120, edgecolor="black", label="Nearest tissue pt")

    # Plot tissue contour
    for contour in contours:
        contour_x = grid_x[(contour[:, 1]).astype(int)]
        contour_y = grid_y[(contour[:, 0]).astype(int)]
        plt.plot(contour_x, contour_y, color="red", linewidth=2, label="Tissue border")

    plt.title(f"m/z {mz_value:.2f}", fontsize=20)
    plt.xlabel("x", fontsize=18)
    plt.ylabel("y", fontsize=18)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.gca().set_aspect('equal')
    plt.gca().invert_yaxis()
    plt.tight_layout()

    # Save or show
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()


In [24]:
df_metrics

,mz,com_distance_to_tissue,mean_intensity_tissue,mean_intensity_bg,bg_dist_max,bg_dist_mean,bg_nonzero_area,weighted_com_x_tissue,weighted_com_y_tissue,weighted_com_x_all,weighted_com_y_all
0,136.0608,2.300358,0.001537,0.000148,18.681542,3.864543,1218,64.412373,51.635506,63.799700,49.418238
1,161.1338,3.484214,0.000762,0.000078,38.327536,10.043459,1616,64.302941,57.714806,64.550293,61.190229
2,179.0362,5.754140,0.000354,0.000281,46.615448,14.223815,7967,64.225191,54.522530,66.552952,59.784816
3,180.0717,4.412945,0.002764,0.000704,46.615448,11.377871,5887,64.332236,56.442829,64.480369,60.853287
4,184.0720,0.756633,0.018103,0.001236,24.596748,3.651858,1196,64.464144,55.372458,64.212772,54.658802
...,...,...,...,...,...,...,...,...,...,...,...
405,1520.1588,0.495449,0.000504,0.000021,18.439089,2.009048,395,64.909398,54.433059,64.423878,54.531750
406,1521.1633,0.326859,0.000450,0.000016,24.758837,1.746905,319,64.834798,54.838868,64.511475,54.886817
407,1522.1682,0.381622,0.000486,0.000019,25.079872,2.164880,369,64.776835,55.136537,64.402857,55.060536
408,1542.1475,0.493767,0.000398,0.000009,23.769729,1.890256,225,64.317162,53.303058,63.839994,53.176108


In [25]:
def score_mz_features(df_metrics, area_weight=0.95):
    # Normalize metrics
    # Normalize distance metrics
    df_metrics["normalized_bg_dist_mean"] = (
        df_metrics["bg_dist_mean"] - df_metrics["bg_dist_mean"].min()
    ) / (df_metrics["bg_dist_mean"].max() - df_metrics["bg_dist_mean"].min())

    # Normalize area metrics
    df_metrics["normalized_bg_nonzero_area"] = (
        df_metrics["bg_nonzero_area"] - df_metrics["bg_nonzero_area"].min()
    ) / (df_metrics["bg_nonzero_area"].max() - df_metrics["bg_nonzero_area"].min())

    # Calculate dispersion score
    # The dispersion score is a weighted combination of the normalized metrics
    # Higher score means better dispersion (lower distance, higher area)
    # Note: area_weight controls the balance between area and distance
    df_metrics["dispersion_score"] = (
        area_weight * df_metrics["normalized_bg_nonzero_area"]
        + (1 - area_weight) * df_metrics["normalized_bg_dist_mean"]
    )
    
    return df_metrics


In [26]:
area_weight = 0.95
# Score m/z features based on dispersion metrics
# This will add a new column 'dispersion_score' to df_metrics
df_metrics = score_mz_features(df_metrics, area_weight=area_weight)
df_metrics

,mz,com_distance_to_tissue,mean_intensity_tissue,mean_intensity_bg,bg_dist_max,bg_dist_mean,bg_nonzero_area,weighted_com_x_tissue,weighted_com_y_tissue,weighted_com_x_all,weighted_com_y_all,normalized_bg_dist_mean,normalized_bg_nonzero_area,dispersion_score
0,136.0608,2.300358,0.001537,0.000148,18.681542,3.864543,1218,64.412373,51.635506,63.799700,49.418238,0.194811,0.129792,0.133043
1,161.1338,3.484214,0.000762,0.000078,38.327536,10.043459,1616,64.302941,57.714806,64.550293,61.190229,0.615024,0.172414,0.194544
2,179.0362,5.754140,0.000354,0.000281,46.615448,14.223815,7967,64.225191,54.522530,66.552952,59.784816,0.899319,0.852538,0.854877
3,180.0717,4.412945,0.002764,0.000704,46.615448,11.377871,5887,64.332236,56.442829,64.480369,60.853287,0.705774,0.629792,0.633591
4,184.0720,0.756633,0.018103,0.001236,24.596748,3.651858,1196,64.464144,55.372458,64.212772,54.658802,0.180346,0.127436,0.130082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,1520.1588,0.495449,0.000504,0.000021,18.439089,2.009048,395,64.909398,54.433059,64.423878,54.531750,0.068623,0.041658,0.043006
406,1521.1633,0.326859,0.000450,0.000016,24.758837,1.746905,319,64.834798,54.838868,64.511475,54.886817,0.050795,0.033519,0.034383
407,1522.1682,0.381622,0.000486,0.000019,25.079872,2.164880,369,64.776835,55.136537,64.402857,55.060536,0.079221,0.038873,0.040891
408,1542.1475,0.493767,0.000398,0.000009,23.769729,1.890256,225,64.317162,53.303058,63.839994,53.176108,0.060544,0.023453,0.025307


In [27]:
df_metrics_filtered = df_metrics[["mz", "dispersion_score", "bg_nonzero_area","normalized_bg_nonzero_area", "bg_dist_mean", "normalized_bg_dist_mean","mean_intensity_tissue", "mean_intensity_bg"]]
df_metrics_filtered.to_csv(f"/home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/mz_metrics_11{sample_group}_{sample}.csv", index=False)

In [28]:
output_folder = f"/home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/paper/{sample_group}_{sample}_output_{min_background_intensity}_score_{area_weight}"

if os.path.exists(output_folder):
    print(f"⚠️ Warning: Output folder '{output_folder}' already exists. Overwriting.")
else:
    os.makedirs(output_folder, exist_ok=True)

df_sorted = df_metrics.sort_values(by="dispersion_score", ascending=False)
rank_num = 1

# Visualize and save m/z features based on dispersion score
# Loop through sorted m/z values and save visualizations
for mz in df_sorted["mz"]:
    filename = f"{rank_num}_mz_{mz:.4f}.png"
    filepath = os.path.join(output_folder, filename)
    
    custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
        (0.0, "#454545"),   # white
        (0.00000001, "#000000"),  # black
        (0.10, "#000080"),  # navy
        (0.15, "#0000FF"),  # blue
        (0.30, "#8000FF"),  # purple-ish
        (0.45, "#FF0000"),  # red
        (0.60, "#FF8000"),  # orange
        (0.75, "#FFFF00"),  # yellow
        (1.0, "#FFFFFF")   # white
    ])
    
    visualize_mz_feature(adata, df_metrics, df_border, mz_value=mz, cmap=custom_cmap, save_path=filepath)
    print(f"Saved: {filepath}")
    rank_num += 1

Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/paper/Aged_AD_4-3_output_0.1_score_0.95/1_mz_307.0371.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/paper/Aged_AD_4-3_output_0.1_score_0.95/2_mz_499.0256.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/paper/Aged_AD_4-3_output_0.1_score_0.95/3_mz_220.0723.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/paper/Aged_AD_4-3_output_0.1_score_0.95/4_mz_242.0613.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/paper/Aged_AD_4-3_output_0.1_score_0.95/5_mz_328.0020.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/paper/Aged_AD_4-3_output_0.1_score_0.95/6_mz_179.0362.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/results/paper/Aged_AD_4-3_output_0.1_score_0.95/7_mz_278.0479.png
Saved: /home/ajarrah/PhD_Thesis/IMZML Tools/lipid dispersion paper/re